Copyright 2026 Tampere University<br>
This notebook and software was developed for a Tampere University course COMP.CS.320.<br>
This source code is licensed under the MIT license. See LICENSE in the exercise repository root directory.<br>
Author(s): Ville Heikkilä \([ville.heikkila@tuni.fi](mailto:ville.heikkila@tuni.fi))

# COMP.CS.320 Data-Intensive Programming, Exercise 4

This exercise is in three parts.

- Tasks 1-4 are basic tasks of using RDDs with text-based data.
- Tasks 5-7 are similar basic tasks for text data but using DataFrames instead.
- Task 8 is a theory question related to the lecture content.

This is the **Python** version, switch to the Scala version if you want to do the tasks in Scala.

Each task has its own cell(s) for the code. Add your solutions to the cells. You are free to add more cells if you feel it is necessary.<br>
There are cells with test code following most of the tasks that involve producing code.<br>
Example outputs for the tasks can be found in a separate notebook: [Exercise 4 example outputs](https://adb-735222077226806.6.azuredatabricks.net/editor/notebooks/1105039603093200)

At the end of the notebook, there is a question regarding the use of AI or other collaboration when working the tasks.<br>
Please remember to answer the AI question. And finally, don't forget to submit your solutions to Moodle: [Weekly Exercise #4](https://moodle.tuni.fi/mod/assign/view.php?id=4162927)

##### Some resources that can help with the tasks in this exercise:

- The [tutorial notebook](https://adb-735222077226806.6.azuredatabricks.net/editor/notebooks/3816292345602665) from our course
- Chapters 3 and 6 in [Learning Spark, 2nd Edition](https://learning.oreilly.com/library/view/learning-spark-2nd/9781492050032/)
    - There are additional code examples in the related [GitHub repository](https://github.com/databricks/LearningSparkV2).
    - The book related notebooks can be imported to Databricks by choosing `import` in your workspace and using the URL<br> `https://github.com/databricks/LearningSparkV2/blob/master/notebooks/LearningSparkv2.dbc`
- Apache Spark [RDD Programming Guide](https://spark.apache.org/docs/4.0.4/rdd-programming-guide.html)
    - [Python RDD methods](https://spark.apache.org/docs/4.0.4/api/python/reference/api/pyspark.RDD.html) documentation
        - Includes the available pyspark methods for pair RDDs.
- Apache Spark [Datasets and DataFrames](https://spark.apache.org/docs/4.0.4/sql-programming-guide.html#datasets-and-dataframes) documentation
    - [Python DataFrame methods](https://spark.apache.org/docs/4.0.4/api/python/reference/pyspark.sql/dataframe.html) documentation
- [Apache Spark documentation](https://spark.apache.org/docs/4.0.4/sql-ref-functions.html) on all available functions that can be used on DataFrames.<br>
  The full [Spark Python functions API listing](https://spark.apache.org/docs/4.0.4/api/python/reference/pyspark.sql/functions.html) for the functions package might have some additional functions listed that have not been updated in the documentation.

In [0]:
# some imports that might be required in the tasks
import dataclasses
import re

from pyspark.rdd import RDD
from pyspark.sql import DataFrame, Row
from pyspark.sql import functions as F
from pyspark.sql.types import *

## Task 1 - Loading text data into an RDD

In the [Shared container](https://portal.azure.com/#view/Microsoft_Azure_Storage/ContainerMenuBlade/~/overview/storageAccountId/%2Fsubscriptions%2Fe0c78478-e7f8-429c-a25f-015eae9f54bb%2FresourceGroups%2Ftuni-cs320-f2026-rg%2Fproviders%2FMicrosoft.Storage%2FstorageAccounts%2Ftunics320f2026gen2/path/shared/etag/%220x8DE33DC133306E1%22/defaultId//publicAccessVal/None) the folder `exercises/ex4/wiki` contains texts from selected Wikipedia articles in raw text format. \
(included are the article on [Finland](https://en.wikipedia.org/wiki/Finland), and all directly linked articles marked as "Main article")

1. Load all articles into a single RDD, `wikitextsRDD`, with one text line as one element in the RDD.
2. Count the total number of lines in the article collection.
3. Count the number of empty lines and the number of non-empty lines in the article collection.
4. Pick the first 6 non-empty lines from the created RDD into variable `lines6`.

In [0]:

wikitextsRDD: RDD[str] = spark.sparkContext.textFile("wasbs://shared@tunics320f2026gen2.blob.core.windows.net/exercises/ex4/wiki/")

numberOfLinesTotal: int = wikitextsRDD.count()

numberOfEmptyLines: int = wikitextsRDD.filter(lambda line: line == "").count()

numberOfNonEmptyLines: int = wikitextsRDD.filter(lambda line: line != "").count()

lines6: list[str] = wikitextsRDD.filter(lambda line: line != "").take(6)

In [0]:
print(f"The total number of lines:     {numberOfLinesTotal}")
print(f"The number of empty lines:      {numberOfEmptyLines}")
print(f"The number of non-empty lines:  {numberOfNonEmptyLines}")
print("====================================")
# Print the first 120 characters of the first 6 non-empty lines
print(*[line[:120] for line in lines6], sep="\n")

## Task 2 - Counting the number of words

Using the RDD from task 1 as a starting point:

- Create `wordsRdd` where each row contains one word with the following rules for words:
    - The word must have at least one character.
    - The word must not contain any numbers, i.e. the digits 0-9.
- Calculate the total number of words in the article collection using `wordsRDD`.
- Calculate the total number of distinct words in the article collection using the same criteria for the words.

You can assume that words in the same line in the original text are separated from each other in the article collection

- either by a standard space character (` `)
- or by a tab character (`\t`).

In this exercise, you should ignore capitalization, parenthesis, and any otherwise unmentioned punctuation characters. \
I.e., "`word`", "`Word`", "`WORD`", "`word.`", "`(word`", and "`word).`" should all be considered as valid and distinct words for this task.

In [0]:
wordsRDD: RDD[str] = wikitextsRDD.flatMap(lambda line: re.split(r'[ \t]', line)).filter(lambda word: len(word) > 0 and not re.search(r'\d', word))

numberOfWords: int = wordsRDD.count()

numberOfDistinctWords: int = wordsRDD.distinct().count()

In [0]:
print(f"The total number of words not containing digits:          {numberOfWords}")
print(f"The total number of distinct words not containing digits:  {numberOfDistinctWords}")

## Task 3 - Counting word occurrences with RDD

**Part 1:**

- Using the article collection data, create a pair RDD, `wordCountRDD`, where each row contains a distinct word and the count for how many times the word can be found in the collection.
    - Use the same word criteria as in task 2, i.e., ignore words which contain digits.

**Part 2:**

- Using the created pair RDD,
    - find the longest word that occurs at least `100` times in the article collection
    - find what is the most common 11-letter word that is not `represented` and starts with an `r`, and what is its count in the collection

You can assume that there is a unique answer to the tasks in part 2. \
(You can modify the given code and find the word and its count separately if that seems easier for you.)

In [0]:
wordCountRDD: RDD[tuple[str, int]] = wordsRDD.map(lambda word: (word, 1)).reduceByKey(lambda a, b: a + b)

longestWord: str = wordCountRDD.filter(lambda pair: pair[1] >= 100).max(key=lambda pair: len(pair[0]))[0]

commonWord, commonWordCount = wordCountRDD.filter(
        lambda pair: len(pair[0]) == 11
        and pair[0].startswith("r")
        and pair[0] != "represented"
    )\
    .max(key=lambda pair: pair[1])

In [0]:
print("The first three items in wordCountRDD:")
_ = [print(f"word: '{item[0]}', count: {item[1]}") for item in wordCountRDD.take(3)]

In [0]:
print(f"The longest word that appears at least 100 times: '{longestWord}'")
print(f"The most common 11-letter word that is not 'represented' and starts with 'r': '{commonWord}' (appears {commonWordCount} times)")

## Task 4 - Letter occurrences in text

Using the article collection data and considering only text lines that have at least `50` characters, \
create a pair RDD, `lettersRDD`, where each row contains

- the letters from English alphabet (from `a` to `z`) that appear at least `15` times on average on each line,
- and the average number of times each letter appears in a line in the source data.
    - when calculating the average, count both the lowercase and the uppercase characters from the line, \
      i.e., both `a` and `A` should be considered when calculating the average occurrence of `a`

Order the RDD such that the most common letter is given first.

In [0]:
import string
alphabet: str = string.ascii_lowercase


valid_lines_RDD = wikitextsRDD.filter(lambda line: len(line) >= 50)
line_count: int = valid_lines_RDD.count()

lettersRDD: RDD[tuple[str, float]] = (
    valid_lines_RDD
    .flatMap(lambda line: line.lower())
    .filter(lambda letter: letter in alphabet)
    .map(lambda letter: (letter, 1))
    .reduceByKey(lambda a, b: a + b)
    .map(lambda pair: (pair[0], pair[1] / line_count))
    .filter(lambda pair: pair[1] >= 15)
    .sortBy(lambda pair: pair[1], ascending=False)
)



In [0]:
print("The average number of times the letters appear in the source data lines:")
for row in lettersRDD.collect():
    print(f"Letter '{row[0]}': => average: {row[1]}")

## Task 5 - Handling text data with DataFrames

In this task, the same Wikipedia article collection as in the RDD tasks 1-4 is used.<br>
In the [Shared container](https://portal.azure.com/#view/Microsoft_Azure_Storage/ContainerMenuBlade/~/overview/storageAccountId/%2Fsubscriptions%2Fe0c78478-e7f8-429c-a25f-015eae9f54bb%2FresourceGroups%2Ftuni-cs320-f2026-rg%2Fproviders%2FMicrosoft.Storage%2FstorageAccounts%2Ftunics320f2026gen2/path/shared/etag/%220x8DE33DC133306E1%22/defaultId//publicAccessVal/None) the folder `exercises/ex4/wiki` contains the texts from selected Wikipedia articles in raw text format. \
(included are the article on [Finland](https://en.wikipedia.org/wiki/Finland), and all directly linked articles marked as "Main article")

Scala supports Datasets, which are typed DataFrames, and the corresponding Scala task is about using them instead of RDDs to redo tasks 1 and 2.<br>
The Datasets are not supported by pyspark, thus this Python task is about using DataFrames instead.

In Scala `DataFrame` is an alias to `Dataset[Row]` where Row is a generic class that can contain any kind of row.<br>
Instead of the generic DataFrame, data can also be handled with Spark using typed Datasets.

**Part 1:**

- Do task 1 again, but this time using the higher level `DataFrame` instead of the low level `RDD`.
    1. Load all articles into a single `DataFrame`, `wikitextsDF`, with one text line as one element in the DataFrame.
    2. Count the total number of lines in the article collection.
    3. Count the number of empty lines and the number of non-empty lines in the article collection.
    4. Pick the first 6 non-empty lines from the created Dataset into variable `first6Lines`.

**Part 2:**

- Do task 2 again, but this time using the higher level `DataFrame` instead of the low level `RDD`.
- Using the DataFrame from first part of this task as a starting point:
    - Create `wordsDF` where each row contains one word with the following rules for words:
        - The word must have at least one character.
        - The word must not contain any numbers, i.e. the digits 0-9.
    - Calculate the total number of words in the article collection using `wordsDF`.
    - Calculate the total number of distinct words in the article collection using the same criteria for the words.

You can assume that words in the same line in the original text are separated from each other in the article collection

- either by a standard space character (` `)
- or by a tab character (`\t`).

In this exercise, you should ignore capitalization, parenthesis, and any otherwise unmentioned punctuation characters. \
I.e., "`word`", "`Word`", "`WORD`", "`word.`", "`(word`", and "`word).`" should all be considered as valid and distinct words for this task.

In [0]:
wikitextsDF: DataFrame = spark.read.text("wasbs://shared@tunics320f2026gen2.blob.core.windows.net/exercises/ex4/wiki/")

totalLinesInDF: int = wikitextsDF.count()

emptyLinesInDF: int = wikitextsDF.filter(F.col("value") == "").count()

nonEmptyLinesInDF: int = wikitextsDF.filter(F.col("value") != "").count()

first6Lines: list[str] = [row[0] for row in wikitextsDF.filter(F.col("value") != "").take(6)]


In [0]:
wordsDF: DataFrame = wikitextsDF.select(F.explode(F.split(F.col("value"), r"[ \t]+")).alias("word")).filter((F.length("word") >= 1) & (~F.col("word").rlike(r"[0-9]")))

wordsInDataset: int = wordsDF.count()

distinctWordsInDataset: int = wordsDF.distinct().count()

In [0]:
print(f"The total number of lines:     {totalLinesInDF}")
print(f"The number of empty lines:      {emptyLinesInDF}")
print(f"The number of non-empty lines:  {nonEmptyLinesInDF}")
print("====================================")
# Print the first 120 characters of the first 6 non-empty lines
print(*[line[:120] for line in first6Lines], sep="\n")
# Use show(1) to show the first line
wikitextsDF.show(1)

In [0]:
print(f"The total number of words not containing digits:          {wordsInDataset}")
print(f"The total number of distinct words not containing digits:  {distinctWordsInDataset}")

## Task 6 - Counting word occurrences with DataFrame

Do task 3 again, but this time using the higher level `DataFrame` instead of the low level `RDD`.<br>
Also, this time you should use the given data class `WordCount` in the final result instead of `(str, int)` tuple like in task 3 with the RDD.

**Part 1:**

- Using the article collection data, create a `DataFrame`, `wordCountDF`, where each row contains a distinct word and the count for how many times the word can be found in the collection.
    - Use the same word criteria as in task 2, i.e., ignore words which contain digits.

**Part 2:**

- Using the created Dataset,
    - find the longest word that occurs at least `100` times in the article collection
    - find what is the most common 11-letter word that is not `represented` and starts with an `r`, and what is its count in the collection
        - The straightforward way is to fully utilize the case class to determine the word and its count together. \
          However, you can determine the word and its count separately if you want, but give the result as an instance of the case class.

In [0]:
@dataclasses.dataclass(frozen=True)
class WordCount:
    word: str
    count: int

In [0]:
wordCountDF: DataFrame =  wordsDF.groupBy("word").count()

longestWordFromDF: str = wordCountDF\
    .filter(F.col("count") >= 100)\
    .orderBy(F.length("word").desc(), F.col("count").desc())\
    .first()[0]

askedWordCount = wordCountDF\
    .filter(
        (F.length("word") == 11)
        & (F.col("word").startswith("r"))
        & (F.col("word") != "represented")
    )\
    .orderBy(F.col("count").desc())\
    .first()

askedWordCount: WordCount = WordCount(word=askedWordCount["word"], count=askedWordCount["count"])


In [0]:
print("The first three items in wordCountDF:")
_ = [print(f"word: '{item['word']}', count: {item['count']}") for item in wordCountDF.take(3)]

In [0]:
print(f"The longest word that appears at least 100 times: '{longestWordFromDF}'")
print(f"The most common 11-letter word that is not 'represented' and starts with 'r': '{askedWordCount.word}' (appears {askedWordCount.count} times)")

## Task 7 - Letter occurrences in text with DataFrames

Do task 4 again, but this time using the higher level `Dataset` instead of the low level `RDD`.<br>

Using the article collection data and considering only text lines that have at least `50` characters, \
create a `DataFrame`, `lettersDF`, where each row contains

- the letters from English alphabet (from `a` to `z`) that appear at least `15` times on average on each line,
- and the average number of times each letter appears in a line in the source data.
    - when calculating the average, count both the lowercase and the uppercase characters from the line, \
      i.e., both `a` and `A` should be considered when calculating the average occurrence of `a`

Order the DataFrame such that the most common letter is given first. \
Finally, collect the data from `lettersDF` to a list of data class `LetterAverage` instances, `letterAverages`.

In [0]:
@dataclasses.dataclass(frozen=True)
class LetterAverage:
    letter: str
    average: float

In [0]:
lettersDF: DataFrame = (
    wikitextsDF.filter(F.length("value") >= 50)
    .select(F.explode(F.split(F.lower("value"), "")).alias("letter"))
    .filter(F.col("letter").rlike("^[a-z]$"))
    .groupBy("letter")
    .agg((F.count("*") / wikitextsDF.filter(F.length("value") >= 50).count()).alias("average"))
    .filter(F.col("average") >= 15)
    .orderBy(F.col("average").desc())
)

letterAverages: list[LetterAverage] = [LetterAverage(r["letter"], r["average"]) for r in lettersDF.collect()]

In [0]:
print("The average number of times the letters appear in the source data lines:")
for letterAverage in letterAverages:
    print(f"Letter '{letterAverage.letter}': => average: {letterAverage.average}")

## Task 8 - Theory questions

Using your own words, answer the following questions:

Spark provides both low-level APIs (RDDs) and high-level APIs (DataFrames and Datasets) for data processing.

1. When should you prefer using RDDs over DataFrames/Datasets?
2. And when should you prefer using DataFrames/Datasets over RDDs?
3. What similarities are can be observed when comparing working with RDDs to working with DataFrames/Datasets?
4. And what are the main differences when comparing RDDs to DataFrames/Datasets?

Extensive answers are not required here.<br>
If your answers do not fit into one screen, you have likely written more than what was expected.

1. RDD should be used when there is so much data that it cannot all fit into memory at the same time, RDD helps splits and distributes the data across computing nodes for them to work on the parts separately
2. DFs allow for easier applying of functions, and faster than RDDs because the program does not have to wait to distribute information across the cluster and collect the data when needed.
3. DFs and RDDs both have map() and filter().
4. Applying functions to RDDs need to write in terms of lambda and native python functions, and DFs you write with spark.sql functions.

## Use of AI and collaboration

Using AI and collaborating with other students is allowed when doing the weekly exercises.<br>
However, the AI use and collaboration should be documented.

- Did you use AI tools while doing this exercise?
  - Did they help? And how did they help?
- Did you work with other students to complete the tasks?
  - Only extensive collaboration is expected to be reported. If you only got help for a couple of the tasks, you don't need to report it here.

???